In [36]:
import httpx
import json
import time
from pathlib import Path
from collections import defaultdict
from narrative_llm_agent.kbase.clients.workspace import Workspace

# JGI Data Portal downloading

This notebook is used to do a mix of fetching and downloading data from the JGI data portal and IMG/M databases.

Relevant links:
* IMG/M - https://img.jgi.doe.gov/cgi-bin/mer/main.cgi
* JGI Data portal - https://data.jgi.doe.gov/

There are some steps not captured in this notebook that resulted in some metadata that's also not present in this repo. 
Instructions are given to rebuild it from IMG/M and the JGI data portal.

# Overview and steps
The goal is to download sets of reads from the JGI data portal that satisfy the following requirements:
* They are public
* They come from bacterial isolates
* They belong to projects in "permanent draft" status - that is, projects that have a draft result but are not expected to have much more analysis done to them
* They are unpublished in either Pubmed or Genbank to demonstrate that we can use the Narrative Research Agent to complete the process

Here are the steps we'll follow to do the downloading.
1. Filter metadata in IMG/M. This will provide us the `ITS_SP_IDs` (sequencing project ids or SP ids) we need.
2. Download the filtered table.
3. Load that table into memory, keep only those rows with no pubmed or genbank id.
4. Ensure that we only have rows with a single unique SP id, to try to avoid having duplicate data objects.
5. Use those SP ids to get project and file metadata from the JGI Data Portal.
6. Extract information from there to form a file request query.
7. Make the query and wait for the download to become available.

## 1. Get metadata from IMG/M
Go do the IMG/M database https://img.jgi.doe.gov/cgi-bin/mer/main.cgi.  
* You'll need to create an account if necessary and log in.
* In the "IMG Content" panel on the left, find the "Bacteria" row and click on the number under "JGI". This will give a list of all ~30,000 JGI-sequenced bacterial sequencing projects.
* Scroll to the bottom. Under "Table Configuration" select the following buttons to add their columns to the table:
  * Under "Metadata" -> "Project Info" select "Is Published" and "Pubmed ID"
  * Under "Metadata" -> "NCBI Metadata" select "NCBI GenBank ID",
  * Under "JGI Specific Field" select "JGI Project ID / ITS SP ID", and "ITS SP ID"
* Hit the "Redisplay" button and the new table should appear.
* Each column has a textbox at top for filtering. Add the following:
  * In Sequencing Status put "Permanent Draft"
  * In GOLD Analysis Project Type put "Isolate"
* At the bottom of the table hit the "Select All" button and "Export" to download a tsv file with everything.
* The following code cells will parse and filter it.

In [17]:
img_data_file = Path("img_data.tsv")

# rows with no pubmed id
no_pmid = []
# JGI Project ids
studies = defaultdict(int)
# ITS AP Ids (analysis project)
its_ap_ids = defaultdict(int)
# ITS SP IDs (sequencing project)
its_sp_ids = defaultdict(int)
# taxon_oids
tax_ids = defaultdict(int)
with open(img_data_file) as infile:
    header_line = infile.readline()
    for line in infile.readlines():
        values = line.rstrip("\n\r").split("\t")
        # we want rows that: no pubmed id or genbank id, are Isolate projects, and in Permanent Draft status
        if not values[14] and not values[15] and "isolate" in values[7].lower() and "permanent draft" in values[2].lower():
            no_pmid.append(values)
            studies[values[23]] += 1
            its_ap_ids[values[24]] += 1
            its_sp_ids[values[25]] += 1
            tax_ids[values[0]] += 1

print(f"rows with no Pubmed ID: {len(no_pmid)}")
print(f"number of unique studies: {len(studies)}")
print(f"number of unique SP IDs: {len(its_sp_ids)}")

rows with no Pubmed ID: 4556
number of unique studies: 310
number of unique SP IDs: 4533


In [13]:
# show index of all columns

header = header_line.strip("\n\r").split("\t")
for idx, txt in enumerate(header):
    print(f"{idx}\t{txt}")

0	taxon_oid
1	NCBI Domain
2	Sequencing Status
3	Study Name
4	Genome Name / Sample Name
5	Sequencing Center
6	IMG Genome ID
7	GOLD Analysis Project Type
8	High Quality
9	Has Coverage
10	Is Public
11	Award DOI
12	Funding Year
13	Is Published
14	Pubmed ID
15	NCBI GenBank ID
16	Sequencing Depth
17	Sequencing Method
18	Sequencing Quality
19	Sequencing Status
20	Sequencing Strategy
21	Genome Size  * assembled
22	Gene Count  * assembled
23	JGI Project ID / ITS SP ID
24	ITS AP ID
25	ITS SP ID
26	ITS Proposal ID


In [239]:
spid_2_row = {}
for row in no_pmid:
    spid_2_row[row[25]] = {"tax_id": row[0], "sp_name": row[4], "img_genome_id": row[6]}

In [78]:
# some rows don't have SP ids, some SP ids are duplicated. Skip these.
sorted(its_sp_ids.items(), key=lambda x: x[1], reverse=True)

[('', 18),
 ('1020215', 2),
 ('1269118', 2),
 ('1284500', 2),
 ('1139853', 2),
 ('1147427', 2),
 ('1020212', 2),
 ('1292578', 1),
 ('1352244', 1),
 ('1025314', 1),
 ('1352100', 1),
 ('1340064', 1),
 ('1352570', 1),
 ('1340081', 1),
 ('1352135', 1),
 ('1551573', 1),
 ('1401606', 1),
 ('1401421', 1),
 ('1306885', 1),
 ('1392001', 1),
 ('1214814', 1),
 ('1357722', 1),
 ('1413023', 1),
 ('1292638', 1),
 ('1030956', 1),
 ('1381299', 1),
 ('1381348', 1),
 ('1413639', 1),
 ('1357681', 1),
 ('1292660', 1),
 ('1357828', 1),
 ('1340402', 1),
 ('1551314', 1),
 ('1347379', 1),
 ('1551413', 1),
 ('1401642', 1),
 ('1219219', 1),
 ('1385549', 1),
 ('1214796', 1),
 ('1351990', 1),
 ('1317087', 1),
 ('1352445', 1),
 ('1355818', 1),
 ('1360769', 1),
 ('1401521', 1),
 ('1357648', 1),
 ('1352535', 1),
 ('1307022', 1),
 ('1401783', 1),
 ('1385585', 1),
 ('1413440', 1),
 ('1413358', 1),
 ('1361092', 1),
 ('1381315', 1),
 ('1401476', 1),
 ('1340347', 1),
 ('1361132', 1),
 ('1340214', 1),
 ('1381045', 1),
 ('

In [19]:
# gather the SP ids that show up in a single row here
singleton_its_sps = [i[0] for i in sorted(its_sp_ids.items(), key=lambda x: x[1], reverse=True)[7:]]
singleton_its_sps[0]

'1292578'

In [110]:
# This is how many sequencing project ids we have that match our criteria.
len(singleton_its_sps)

4526

## 2. Get File metadata from the JGI Data Portal
See the Data Portal documentation for details. But here's the summary:

Reads data in JGI isn't always available. After some time, it gets purged from the main system and backed up to a permanent storage tape drive, which need a specific request for download and about a 24 hour wait while it gets restored.

There are a few steps to get that.
1. Using the SP ids, fetch the full project and file metadata from the data portal.
2. Use those to get only FASTQ files (reads-containing files).
3. Make a request to download those files from the data portal.

The following code cells walk through that process for the first 100 SP IDs, `spids_of_interest`.

In [20]:
def query_maker(its_sp_id: str) -> str:
    """
    Writes a query to get file metadata from JGI for a single SP ID.
    """
    base_url = "https://files.jgi.doe.gov"
    return base_url + f"/img_file_list/?its_sp_id={its_sp_id}&api_version=2&a=false&h=false&d=asc&p=1&x=10&t=simple&ff[file_type]=fastq"

def run_query(its_sp_id: str) -> dict:
    """
    Write and run a query to fetch metadata for a single SP ID and return it as a dictionary.
    """
    resp = httpx.get(query_maker(its_sp_id))
    return resp.json()

In [21]:
# Select the first 100 SP IDs.
spids_of_interest = singleton_its_sps[:100]

In [116]:
# Pull the metadata for each SPID of interest and store it locally as JSON.
# Note that running queries too fast might cause some throttling and timeout.
# (Bill: I haven't seen this happen with fewer than ~300 queries in a row, but
# adding a time.sleep(5) after each 100 or so seems to help.)
for spid in spids_of_interest:
    meta_result = run_query(spid)
    with open(f"{spid}_metadata.json", "w") as out_meta:
        json.dump(meta_result, out_meta, indent=4)

## 3. Build a file request packet to send to the JGI File server.
See here for details: https://files.jgi.doe.gov/apidoc/#/POST/request_archived_files_create

Short version:
* For each metadata file, build an aggregate id from the "agg" and "agg_id" fields. Will look like: `IMG_SP-1352244`
* Get all file ids (`_id` field of each file object) and put them in a list.
* Put all this together in a structure. E.g.:
```
{
    "ids": {
        "IMG_SP-123456": {
            "file_ids": ["asdfasdf", "fdsafdsa"]
        },
        "IMG_SP-789012": {
            "file_ids": ["qwerqwer", "rewqrewq"]
        }
    }
}
```

This becomes the `file_request_packet`

For this, you'll also need the Single sign-on token. This is available from the Account dropdown menu in the upper-right of the main data portal page. Select "Copy my session token" to get it.

In [24]:
# some simplifying housekeeping
# map file names -> img_sp_ids
file_2_spid = {}
file_2_filename = {}
filename_2_spid = {}
# map spid to file(s)
spid_2_files = {}
file_data = []

file_request_query = {"ids": {}}
for spid in spids_of_interest[:100]:
    print(spid)
    with open(f"{spid}_metadata.json", "r") as in_meta:
        meta = json.load(in_meta)
        if "organisms" not in meta:
            print(f"{spid} no organisms key found")
            continue
        if (len(meta["organisms"]) == 0):
            print(f"{spid} no organisms found")
            continue
        if "files" not in meta["organisms"][0] or len(meta["organisms"][0]["files"]) == 0:
            print(f"{spid} no fastq files found")
            continue
        agg_id = (f'{meta["organisms"][0]["agg"]}-{meta["organisms"][0]["agg_id"]}')
        file_ids = []
        for data_file in meta["organisms"][0]["files"]:
            file_ids.append(data_file["_id"])
            file_2_filename[data_file["_id"]] = data_file["file_name"]
            filename_2_spid[data_file["file_name"]] = spid
            file_2_spid[data_file["_id"]] = spid
        file_request_query["ids"][agg_id] = {"file_ids": file_ids}
        spid_2_files[spid] = file_ids
        file_data.append(data_file)

1292578
1352244
1025314
1352100
1340064
1352570
1340081
1352135
1551573
1401606
1401421
1306885
1392001
1214814
1357722
1413023
1292638
1030956
1381299
1381348
1413639
1357681
1292660
1357828
1340402
1551314
1347379
1551413
1401642
1219219
1385549
1214796
1351990
1317087
1352445
1355818
1360769
1401521
1357648
1352535
1307022
1401783
1385585
1413440
1413358
1361092
1381315
1401476
1340347
1361132
1340214
1381045
1413287
1551102
1018461
1018461 no organisms found
1413237
1551217
1270347
1317170
1392030
1352195
1352245
1413662
1551532
1551017
1307026
1357514
1306932
1292629
1381670
1392052
1392148
1340084
1352183
1134423
1551412
1357471
1352097
1392023
1352571
1407839
1401299
1551334
1381444
1551451
1381328
1139671
1155679
1551151
1357731
1551015
1551289
1105609
1352409
1401552
1401352
1214714
1586655
1269079
1340370


In [134]:
file_request_query

{'ids': {'IMG_SP-1292578': {'file_ids': ['6086dda253a8c649b0cb36b8']},
  'IMG_SP-1352244': {'file_ids': ['622f645fe99caa81935b477e']},
  'IMG_SP-1025314': {'file_ids': ['529679d4067c0121bf0d6373',
    '52964be3067c0121bf0d621e']},
  'IMG_SP-1352100': {'file_ids': ['62029526f9c444ffdffcb7e0']},
  'IMG_SP-1340064': {'file_ids': ['62029171f9c444ffdffcb74a']},
  'IMG_SP-1352570': {'file_ids': ['6228c68de99caa81935ae628']},
  'IMG_SP-1340081': {'file_ids': ['620290c8f9c444ffdffcb730']},
  'IMG_SP-1352135': {'file_ids': ['62029384f9c444ffdffcb7a1']},
  'IMG_SP-1551573': {'file_ids': ['6835caf1e69ed90826268275']},
  'IMG_SP-1401606': {'file_ids': ['63a8dcc63b5d0133c7411b0c']},
  'IMG_SP-1401421': {'file_ids': ['63a8dc6a3b5d0133c7411a11']},
  'IMG_SP-1306885': {'file_ids': ['6099791a53a8c649b0cc08a3']},
  'IMG_SP-1392001': {'file_ids': ['636eb350be000c239f764c54']},
  'IMG_SP-1214814': {'file_ids': ['5c13125446d1e64a530fbfca']},
  'IMG_SP-1357722': {'file_ids': ['6269e082f21e5a14d08da48b']},
 

In [146]:
sso_token = "REDACTED"

In [129]:
# Make the file request and get the JSON result.
resp = httpx.post(
    "https://files.jgi.doe.gov/request_archived_files/", 
    json=file_request_query, 
    headers={"Authorization": sso_token}
)

In [130]:
resp.json()

{'updated_count': 1,
 'restored_count': 10,
 'request_id': 525995,
 'request_status_url': 'https://files.jgi.doe.gov/request_archived_files/requests/525995'}

In [131]:
# Check the request status
status_resp = httpx.post(
    "https://files.jgi.doe.gov/request_archived_files/requests/", 
    json={"request_ids": [resp.json()["request_id"]]}, 
    headers={"Authorization": sso_token}
)

In [132]:
status_resp.json()

{'525995': {'file_ids': ['6835caf1e69ed90826268275',
   '62029384f9c444ffdffcb7a1',
   '6228c68de99caa81935ae628',
   '62029171f9c444ffdffcb74a',
   '620290c8f9c444ffdffcb730',
   '62029526f9c444ffdffcb7e0',
   '63a8dcc63b5d0133c7411b0c',
   '6086dda253a8c649b0cb36b8',
   '622f645fe99caa81935b477e',
   '529679d4067c0121bf0d6373',
   '52964be3067c0121bf0d621e'],
  'status': 'pending',
  'expiration_date': '2025-12-18T10:34:07.121276-08:00'}}

## 4. Files have now been requested

They'll be available for download a single packet in about 24 hours.  
Once that's done, they can be moved to KBase in a few ways:
1. Download them to your local computer and upload them separately. This isn't recommended for large datasets.
2. Use Globus to move them from JGI to your KBase Staging Service. See here for help: https://docs.kbase.us/data/globus
3. Use the Data Transfer Service (DTS) to make the connection. See here for help: https://docs.kbase.us/data/jgi-transfer and below for a brief programmatic guide. As of this writing, the DTS has been released in beta, so things might change.

In [25]:
inst = build_instructions(file_data)

In [26]:
inst

{'protocol': 'KBase narrative import',
 'objects': [{'data_type': 'fastq_reads_interleaved',
   'parameters': {'fastq_fwd_staging_file_name': '/sdm/illumina/05/25/08/52508.4.363682.GCCTTGTT-GCCTTGTT.fastq.gz',
    'name': '52508.4.363682.GCCTTGTT-GCCTTGTT.fastq.gz_reads',
    'sequencing_tech': 'Unknown',
    'single_genome': 1,
    'read_orientation_outward': 0,
    'insert_size_std_dev': None,
    'insert_size_mean': None}},
  {'data_type': 'fastq_reads_interleaved',
   'parameters': {'fastq_fwd_staging_file_name': '/sdm/illumina/05/26/64/52664.1.415157.TTGGACGT-TTGGACGT.fastq.gz',
    'name': '52664.1.415157.TTGGACGT-TTGGACGT.fastq.gz_reads',
    'sequencing_tech': 'Unknown',
    'single_genome': 1,
    'read_orientation_outward': 0,
    'insert_size_std_dev': None,
    'insert_size_mean': None}},
  {'data_type': 'fastq_reads_interleaved',
   'parameters': {'fastq_fwd_staging_file_name': '/sdm/illumina/00/74/79/7479.7.74223.AGTTCC.fastq.gz',
    'name': '7479.7.74223.AGTTCC.fastq.gz

Some code below to redo file downloads to fill in for datasets that failed.

In [175]:
num_unpurged_files = 10
total = len(singleton_its_sps[2216:])
current = 0
for idx, spid in enumerate(singleton_its_sps[2216:]):
    print(f"{idx}/{total} {spid}")
    meta_result = run_query(spid)
    # find some metadata that hasn't been purged. Just a few. Please....
    if len(meta_result.get("organisms", [])) and len(meta_result["organisms"][0].get("files", [])):
        for file_info in meta_result["organisms"][0]["files"]:
            is_purged = file_info["file_status"].lower() == "purged"
            if not is_purged:
                print(f"\tGOT ONE! {spid}")
                spids_to_fetch.append(spid)
                with open(f"./jgi_metadata/{spid}_metadata.json", "w") as out_meta:
                    json.dump(meta_result, out_meta, indent=4)
    if len(spids_to_fetch) >= num_unpurged_files:
        break
    if idx % 100 == 0:
        print("taking a break for 5 seconds")
        time.sleep(5)
        print("break's over!")

0/2310 1306926
taking a break for 5 seconds
break's over!
1/2310 1134408
2/2310 1360795
3/2310 1413781
4/2310 1381698
5/2310 1551231
6/2310 1134407
7/2310 1357517
8/2310 1139666
9/2310 1292635
10/2310 1392081
11/2310 1392117
12/2310 1392109
13/2310 1381511
14/2310 1352387
15/2310 1401261
16/2310 1269061
17/2310 1551274
18/2310 1214731
19/2310 1413585
20/2310 1413602
21/2310 1340198
22/2310 1317446
23/2310 1360997
24/2310 1381058
25/2310 1361088
26/2310 1551395
27/2310 1351885
28/2310 1347388
29/2310 1381458
30/2310 1306997
31/2310 1401329
32/2310 1412945
33/2310 1352417
34/2310 1361086
35/2310 1282091
36/2310 1413044
37/2310 1360765
38/2310 1413452
39/2310 1326024
40/2310 1317117
41/2310 1401908
42/2310 1381355
43/2310 1550951
44/2310 1352273
45/2310 1214693
46/2310 1360992
47/2310 1401300
48/2310 1352382
49/2310 1214775
50/2310 1326097
51/2310 1381443
52/2310 1219235
53/2310 1551467
54/2310 1413000
55/2310 1340268
56/2310 1134414
57/2310 1551501
58/2310 1352397
59/2310 1360762
60/2310

In [179]:
spids_to_fetch

['1248680',
 '1030857',
 '1030857',
 '1053055',
 '1227824',
 '1248546',
 '1347351',
 '1186026',
 '1147911',
 '1186048']

In [181]:
revised_spids_to_fetch = spids_to_fetch[0:1] + spids_to_fetch[2:]
revised_spids_to_fetch

['1248680',
 '1030857',
 '1053055',
 '1227824',
 '1248546',
 '1347351',
 '1186026',
 '1147911',
 '1186048']

In [190]:
# some simplifying housekeeping
# map file names -> img_sp_ids
new_file_2_spid = {}
new_file_2_filename = {}
new_filename_2_spid = {}
# map spid to file(s)
new_spid_2_files = {}
new_file_data = []

new_file_request_query = {"ids": {}}
for spid in revised_spids_to_fetch:
    print(spid)
    with open(f"./jgi_metadata/{spid}_metadata.json", "r") as in_meta:
        meta = json.load(in_meta)
        if "organisms" not in meta:
            print(f"{spid} no organisms key found")
            continue
        if (len(meta["organisms"]) == 0):
            print(f"{spid} no organisms found")
            continue
        if "files" not in meta["organisms"][0] or len(meta["organisms"][0]["files"]) == 0:
            print(f"{spid} no fastq files found")
            continue
        agg_id = (f'{meta["organisms"][0]["agg"]}-{meta["organisms"][0]["agg_id"]}')
        file_ids = []
        for data_file in meta["organisms"][0]["files"]:
            print(f"{data_file['_id']} - {data_file['file_status']}")
            if data_file["file_status"] != "RESTORED":
                continue
            file_ids.append(data_file["_id"])
            new_file_2_filename[data_file["_id"]] = data_file["file_name"]
            new_filename_2_spid[data_file["file_name"]] = spid
            new_file_2_spid[data_file["_id"]] = spid
            new_file_data.append(data_file)
        new_file_request_query["ids"][agg_id] = {"file_ids": file_ids}
        new_spid_2_files[spid] = file_ids


1248680
67ae064768dd0c5de8e12ada - RESTORED
1030857
5329352d49607a1be00599a1 - RESTORED
5329760c49607a1be0059a5c - PURGED
1053055
545d865e0d87855284890b94 - PURGED
545d5e010d87855284890b40 - RESTORED
1227824
67c631d367ef7b237e943332 - BACKUP_COMPLETE
1248546
67d0973ed72b25923552b58d - RESTORED
1347351
6267f6a1f21e5a14d08d81ce - RESTORED
6268352cf21e5a14d08d89f5 - PURGED
1186026
67c6318d67ef7b237e9432a9 - BACKUP_COMPLETE
1147911
67c6318c67ef7b237e94329d - BACKUP_COMPLETE
1186048
67d0972fd72b25923552b524 - RESTORED


In [191]:
new_file_request_query

{'ids': {'IMG_SP-1248680': {'file_ids': ['67ae064768dd0c5de8e12ada']},
  'IMG_SP-1030857': {'file_ids': ['5329352d49607a1be00599a1']},
  'IMG_SP-1053055': {'file_ids': ['545d5e010d87855284890b40']},
  'IMG_SP-1227824': {'file_ids': []},
  'IMG_SP-1248546': {'file_ids': ['67d0973ed72b25923552b58d']},
  'IMG_SP-1347351': {'file_ids': ['6267f6a1f21e5a14d08d81ce']},
  'IMG_SP-1186026': {'file_ids': []},
  'IMG_SP-1147911': {'file_ids': []},
  'IMG_SP-1186048': {'file_ids': ['67d0972fd72b25923552b524']}}}

In [194]:
new_file_2_spid.keys()

dict_keys(['67ae064768dd0c5de8e12ada', '5329352d49607a1be00599a1', '545d5e010d87855284890b40', '67d0973ed72b25923552b58d', '6267f6a1f21e5a14d08d81ce', '67d0972fd72b25923552b524'])

In [193]:
build_instructions(new_file_data)

{'protocol': 'KBase narrative import',
 'objects': [{'data_type': 'fastq_reads_interleaved',
   'parameters': {'fastq_fwd_staging_file_name': '/sdm/illumina/05/30/90/53090.2.581637.CCACTCGAGC-AGGACTCTTC.fastq.gz',
    'name': '53090.2.581637.CCACTCGAGC-AGGACTCTTC.fastq.gz_reads',
    'sequencing_tech': 'Unknown',
    'single_genome': 1,
    'read_orientation_outward': 0,
    'insert_size_std_dev': None,
    'insert_size_mean': None}},
  {'data_type': 'fastq_reads_interleaved',
   'parameters': {'fastq_fwd_staging_file_name': '/sdm/illumina/00/77/79/7779.3.83550.AAGCGA.fastq.gz',
    'name': '7779.3.83550.AAGCGA.fastq.gz_reads',
    'sequencing_tech': 'Unknown',
    'single_genome': 1,
    'read_orientation_outward': 0,
    'insert_size_std_dev': None,
    'insert_size_mean': None}},
  {'data_type': 'fastq_reads_interleaved',
   'parameters': {'fastq_fwd_staging_file_name': '/sdm/illumina/00/84/65/8465.8.102013.GACGAC.fastq.gz',
    'name': '8465.8.102013.GACGAC.fastq.gz_reads',
    'se

End file download redo code

## (optional) Transferring to KBase via DTS
This is a little out of scope here, as the runner script can work from both local files and KBase data objects. But here's the process.

There is a Python client for the DTS available at https://github.com/kbase/dtspy
The DTS moves files from JGI to KBase, and provides a structured `manifest.json` file. This file can optionally include instructions for the KBase file importer to use in transforming data files to operable KBase data objects. The following functions build the instructions for importing interleaved paired-end reads from a FASTQ file, and for starting the file copying process.

Note that this is included for completeness and reproducibility, but the `dtspy` module is not installed in the KBase Research Agent repo.

Note also that making a DTS request to copy data from JGI to KBase requires a KBase account linked to an ORCID, and you'll need a KBase authentication token along with the ORCID.

In [32]:
def build_instructions(file_data: list[dict]) -> dict:
    """
    Build a set of instructions to pass to the DTS for build a manifest to import to KBase.
    Converts the file list to expected import paths so that the "Import from DTS manifest" function
    can be used.
    """
    obj_set = []

    for file_info in file_data:
        file_path = Path(file_info["file_path"].split("dm_archive")[-1][1:]) / file_info["file_name"]
        obj_name = file_info["file_name"] + "_reads"
        obj_set.append({
            "data_type": "fastq_reads_interleaved",
            "parameters": {            
                "fastq_fwd_staging_file_name": str(file_path),
                "name": obj_name,
                "sequencing_tech": "Unknown",
                "single_genome": 1,
                "read_orientation_outward": 0,
                "insert_size_std_dev": None,
                "insert_size_mean": None
            }
        })
    
    return {
        "protocol": "KBase narrative import",
        "objects": obj_set
    }

In [35]:
def make_dts_request(orcid: str, kbase_token: str, file_ids: list, instructions: dict=None):
    files_to_request = [f"JDP:{file_id}" for file_id in file_ids]
    dts_client = dts.Client(api_key=kbase_token,
                            server="https://dts.kbase.us")
    xfer_id = dts_client.transfer(orcid=orcid,
                                 file_ids=files_to_request,
                                 source='jdp',
                                 destination='kbase',
                                 instructions=instructions)
    print(f'Transfer ID: {xfer_id}')

# Building result table

This last section describes how the `img_llm_annotations.tsv` file was made.  
The predicted GTDB additions are made via the script in `scripts/generate_gtdb_annotations.py`.  
The remainder of this notebook builds the table with columns:

* JGI sequencing project id
* IMG genome id
* File id
* File name
* Original upa
* Generated narrative id
* Narrative url
* Species name

Note that for this project, all reads were imported into the narrative with id 239038 (i.e. https://narrative.kbase.us/narrative/239038)

In [195]:
# Need to make a table - should have
# * spid x
# * taxon_oid 
# * species name
# * file name
# * original UPA
# * narrative it was run in
# * predicted species

How to map:  
Not all files are used or downloaded, not all spids were used.  
Can start with UPA and filename?

Run list_objects on ws 239038? = get UPA, get object name. strip "_reads" off name = file name

We have file_data and new_file_data (merge them!)  
this gets us file name and id (file_name, _id)

we have file_2_spid and new file_2_spid (merge those too!) - gets us the spid  
spid gets us metadata, so we get taxon oid, name from img_data.tsv file

Just need the narrative id - can parse out of logs  
And predicted spp from final run... we'll get there next. Maybe LLM help.

In [213]:
ws = Workspace(token="NOPE")
reads_objs = ws.list_workspace_objects(239038, as_dict=True)

In [224]:
reads_objs[3]

{'ws_id': 239038,
 'obj_id': 5,
 'version': 1,
 'name': '52659.4.414109.CCAGTGTT-CCAGTGTT.fastq.gz_reads',
 'ws_name': 'wjriehl:narrative_1764881155555',
 'type': 'KBaseFile.PairedEndLibrary-2.1',
 'saved': '2025-12-08T23:06:11+0000',
 'saved_by': 'wjriehl',
 'size_bytes': 760,
 'metadata': None,
 'path': [],
 'upa': '239038/5/1'}

In [226]:
filename_2_spid[reads_objs[2]["name"].rstrip("_reads")]

'1551573'

In [229]:
# new file data was taken from the revised block above that's minified
all_file_data = file_data + new_file_data

In [290]:
# compress all the data into a dictionary keyed on the original UPA.
final_table_by_upa = {}
for obj in reads_objs[2:]:
    upa = obj["upa"]
    obj_name = obj["name"]
    file_name = obj["name"].rstrip("_reads")
    print(file_name)
    file_info = next((f for f in all_file_data if f["file_name"] == file_name), {})
    print(file_info)
    spid = str(file_info["metadata"]["sequencing_project_id"])
    if spid.startswith("[") and spid.endswith("]"):
        spid = spid[1:-1]
    print(spid)
    meta_row = spid_2_row[spid]
    final_table_by_upa[upa] = {
        "upa": upa,
        "obj_name": obj_name,
        "file_name": file_name,
        "spid": spid,
        "file_id": file_info["_id"],
    } | meta_row

53127.1.597830.TGAGGTAAGA-CTCTGCAAGC.fastq.gz
{'_id': '6835caf1e69ed90826268275', '_highlight': None, 'file_path': '/global/dna/dm_archive/sdm/illumina/05/31/27', 'file_status_id': 10, 'metadata': {'proposal': {'award_doi': '10.46936/10.25585/60008892', 'pi': {'institution': 'Univ of North Dakota', 'country': 'United States', 'email_address': 'kjvenkat1955@gmail.com', 'last_name': 'Venkateswaran', 'contact_id': 50073, 'middle_name': '', 'first_name': 'Kasthuri'}, 'lay_description': 'The Biotechnology and Planetary Protection Group at Jet Propulsion Laboratory (JPL) has been at the forefront of a significant effort focused on studying bacterial and fungal strains from both the International Space Station (ISS) and Mars mission assembly cleanrooms. Over the course of six years and spanning six different missions to the ISS, as well as approximately 20 years for Mars mission cleanrooms, they have successfully isolated and sequenced 3,500 strains.\n\nIn the next phase of this project, we a

In [294]:
# Now, get the narrative id for each one
# This is found in the log files for all runs. Too large for the repo, but available on request.
upa_to_logfile = {}
logfiles = list(filter(lambda f: f.startswith("llm_genome_annotation_239038"), os.listdir("../")))

for upa in final_table_by_upa.keys():
    logfile = next(f for f in logfiles if upa.replace("/", "_") in f)
    upa_to_logfile[upa] = logfile

for upa, logfile in upa_to_logfile.items():
    with open(Path("..") / logfile) as log:
        for line in log.readlines():
            if "created narrative with id" in line:
                narrative_id = line.rstrip().split()[-1]
                print(f"{upa}\t{narrative_id}")
                final_table_by_upa[upa]["narrative_id"] = narrative_id
                final_table_by_upa[upa]["narrative_url"] = f"https://narrative.kbase.us/narrative/{narrative_id}"

239038/4/1	239697
239038/5/1	239700
239038/7/1	239694
239038/8/1	239691
239038/11/1	239696
239038/12/1	239698
239038/13/1	239699
239038/14/1	239692
239038/15/1	239693
239038/20/1	239738
239038/21/1	239740
239038/23/1	239736
239038/28/1	239739
239038/30/1	239741
239038/31/1	239734
239038/32/1	239742
239038/33/1	239735
239038/34/1	239733
239038/38/1	239737
239038/41/1	239789
239038/42/1	239792
239038/43/1	239786
239038/45/1	239752
239038/46/1	239750
239038/47/1	239751
239038/48/1	239746
239038/53/1	239787
239038/55/1	239748
239038/56/1	239753
239038/60/1	239791
239038/62/1	239795
239038/64/1	239793
239038/68/1	239788
239038/69/1	239794
239038/70/1	239790
239038/74/1	239850
239038/75/1	239858
239038/76/1	239855
239038/78/1	239851
239038/79/1	239849
239038/80/1	239854
239038/85/1	239853
239038/87/1	239857
239038/89/1	239852
239038/90/1	239856
239038/93/1	239873
239038/94/1	239865
239038/95/1	239869
239038/96/1	239867
239038/98/1	239872
239038/101/1	239868
239038/103/1	239866
239038/108/1	2

In [297]:
# Name the columns and output the TSV file

cols = [
    "JGI sequencing project id",
    "IMG genome id",
    "File id",
    "File name",
    "Original upa",
    "Generated narrative id",
    "Narrative url",
    "Species name",
]

file_lines = ["\t".join(cols)]
for data in final_table_by_upa.values():
    if "narrative_id" not in data:
        print(data)
    data_row = [
        data["spid"],
        data["img_genome_id"],
        data["file_id"],
        data["file_name"],
        data["upa"],
        data["narrative_id"],
        data["narrative_url"],
        data["sp_name"]
    ]
    file_lines.append("\t".join(data_row))

with open("img_llm_annotations.tsv", "w") as outfile:
    outfile.write("\n".join(file_lines))

In [304]:
# This final step makes all LLM-generated narratives publicly readable.
for data in final_table_by_upa.values():
    ws.make_kbase_jsonrpc_1_call("Workspace.set_global_permission", [{"id": data["narrative_id"], "new_permission": "r"}])